# 6 · Climate scenario table from a sister repository

**Question:** how should a GIS analyst read a climate table before mapping it?

This CSV is copied from your [JupyterLite-zarr-sql-views](https://github.com/jltobias/JupyterLite-zarr-sql-views) repository. It originates in the [C3S Atlas user tools](https://github.com/ecmwf-projects/c3s-atlas) under Apache 2.0. It describes CMIP6 global warming level windows by model and scenario. It is **not** a geographic layer, so no map is drawn until a geographic key and spatial data are supplied.

![Runtime boundaries](https://jltobias.github.io/JupyterLite-RustGIS/assets/runtime-boundaries.svg)

## Parse the unusual two-row header

The first header row holds warming levels; the second holds scenarios. `9999` and `NA` are missing or unavailable markers, not years.

In [ ]:
import csv
from pathlib import Path
with Path('data/CMIP6_WarmingLevels.csv').open(newline='', encoding='utf-8') as handle:
    rows=list(csv.reader(handle))
levels, scenarios = rows[:2]
print('Models:', len(rows)-2, '| pairs:', len(levels)-1)
print([(scenarios[i], levels[i]) for i in range(1,6)])

## Make tidy records

In [ ]:
records=[]
for row in rows[2:]:
    for i in range(1,min(len(row),len(levels))):
        if row[i] not in ('9999','NA',''):
            records.append({'model':row[0],'scenario':scenarios[i],'level':levels[i],'window':row[i]})
print('Usable model–scenario–level windows:',len(records))
print(records[:3])

## Count availability, not climate impact

This chart counts table entries with a reported time window. It does not show probability, temperature magnitude, or regional exposure.

In [ ]:
from collections import Counter
from IPython.display import display, HTML
from viz import bars
counts=Counter(r['scenario'] for r in records)
display(HTML(bars(sorted(counts),[counts[k] for k in sorted(counts)],'Available model windows by scenario',' records')))

## Inspect one warming level

In [ ]:
level='2'
subset=[r for r in records if r['level']==level]
print(f'{len(subset)} records for {level}°C')
for r in subset[:8]: print(r['scenario'],r['model'][:26],r['window'])

## A timeline view

Use the start of each reported window as a small comparative graphic. It is not a forecast interval plot.

In [ ]:
starts=[int(r['window'][:4]) for r in subset if r['window'][:4].isdigit()]
decades=Counter((year//10)*10 for year in starts)
display(HTML(bars([str(d) for d in sorted(decades)],[decades[d] for d in sorted(decades)],'Start decade of reported 2°C windows',' models','#e79c61')))

## GIS decision: can this be mapped?

No. A table of global model windows lacks coordinates, polygons, or a geographic join key. A responsible RustGIS workflow would obtain a regional raster/vector product with its own provenance, then join or compare using a defensible spatial and temporal method.

In [ ]:
has_geometry = any(key in records[0] for key in ('geometry','lon','lat','region_id'))
print('Has geographic key?', has_geometry)
assert not has_geometry

**Challenge:** compare counts for 1.5°C and 3°C, then explain why missingness can bias a visual summary. Do not substitute this count for a climate risk estimate.

**Citations:** [source repository and notices](https://github.com/jltobias/JupyterLite-zarr-sql-views/blob/main/THIRD_PARTY_NOTICES.md), [C3S Atlas user tools](https://github.com/ecmwf-projects/c3s-atlas), [Apache 2.0 license in this repo](https://github.com/jltobias/JupyterLite-RustGIS/blob/main/licenses/c3s-atlas-Apache-2.0.txt).